# 🏋️ Diet & Fitness ML Project
## Shared Notebook — Data Understanding & Preparation

> **Run this notebook once as a team.**  
> It produces `df_clean.csv` which every personal notebook loads as its starting point.

---

## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.impute import SimpleImputer

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 60)

## 2. Load Dataset

In [ ]:
df = pd.read_csv('dataset_Diet.csv', sep=';')
print(f'Shape: {df.shape}')
df.head(3)

## 3. Data Understanding
### 3.1 General Info

In [ ]:
df.info()

In [ ]:
df.describe()

### 3.2 Target columns — value distributions

In [ ]:
print('--- diet_type ---')
print(df['diet_type'].value_counts())
print()
print('--- meal_type ---')
print(df['meal_type'].value_counts())
print()
print('--- Workout_Type ---')
print(df['Workout_Type'].value_counts())

### 3.3 Missing values

In [ ]:
missing_pct = df.isna().mean() * 100
missing_pct = missing_pct[missing_pct > 0].sort_values(ascending=False)
print(f'{len(missing_pct)} columns have missing values:\n')
print(missing_pct)

### 3.4 Column types

In [ ]:
num_cols = df.select_dtypes(include='number').columns
cat_cols = df.select_dtypes(include='object').columns
print(f'Numeric columns ({len(num_cols)}):', list(num_cols))
print()
print(f'Categorical columns ({len(cat_cols)}):', list(cat_cols))

### 3.5 Outlier visualisation (sample columns)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].boxplot(df['Avg_BPM'].dropna(), vert=False)
axes[0].set_title('Avg_BPM')
axes[1].boxplot(df['Physical exercise'].dropna(), vert=False)
axes[1].set_title('Physical exercise')
plt.suptitle('Outlier Detection — Boxplots')
plt.tight_layout()
plt.show()

## 4. Data Preparation
### 4.1 Impute missing values

In [ ]:
# Numeric → median
num_imputer = SimpleImputer(strategy='median')
df[num_cols] = num_imputer.fit_transform(df[num_cols])

# Categorical → most frequent
cat_imputer = SimpleImputer(strategy='most_frequent')
df[cat_cols] = cat_imputer.fit_transform(df[cat_cols])

print('Missing values after imputation:', df.isna().sum().sum())

### 4.2 Remove duplicates

In [ ]:
before = len(df)
df = df.drop_duplicates()
print(f'Removed {before - len(df)} duplicate rows. Rows remaining: {len(df)}')

### 4.3 Remove outliers (IQR method)

In [ ]:
num_cols = df.select_dtypes(include='number').columns  # refresh after drop_duplicates

Q1 = df[num_cols].quantile(0.25)
Q3 = df[num_cols].quantile(0.75)
IQR = Q3 - Q1

outlier_mask = ((df[num_cols] < (Q1 - 1.5 * IQR)) |
                (df[num_cols] > (Q3 + 1.5 * IQR))).any(axis=1)

print(f'Outlier rows detected: {outlier_mask.sum()}')
print(f'Rows before: {len(df)}')

df_clean = df[~outlier_mask].copy()
print(f'Rows after: {len(df_clean)}')

### 4.4 Final check

In [ ]:
print('Shape:', df_clean.shape)
print('Missing values:', df_clean.isna().sum().sum())
df_clean.info()

## 5. Export clean dataset

> ⚠️ Every teammate's personal notebook starts from this file.

In [ ]:
df_clean.to_csv('df_clean.csv', index=False)
print('✅ df_clean.csv saved successfully.')